# Figure 4

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
image_format = 'pgf'
dpi = 600
resources_path = './resources'
target_path = './images'

In [ ]:
Path(target_path).mkdir(parents=True, exist_ok=True)

In [ ]:
unet = pd.read_csv(
    f'{resources_path}/scores_unet.csv',
    converters=dict(rmse=lambda x: float(x) * 100),  # RMSE from m to cm
)
unet

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))

noises = [
    [0.1, 0.01, 'x'],
    [0.2, 0.02, 'o'],
    [0.3, 0.03, 's'],
]

metrics = ['rmse', 'lambda_']
x = [1, 5, 10, 15]

handles = []
labels = []
save_ = True

# Baselines
labels.append(r'Baseline ($n=1, \sigma=0$)')
handles.append(
    ax[0].plot(  # UNet RMSE
        x, unet.rmse[0] * np.ones_like(x), label='Baseline',
        linestyle='--', color='k', marker='|',
    )[0]
)
ax[1].plot(  # UNet lambda
    x, unet.lambda_[0] * np.ones_like(x), label='Baseline',
    linestyle='--', color='k', marker='|',
)

# Scores
for i, metric in enumerate(metrics):
    for noise in noises:
        h_, = ax[i].plot(
            x,
            unet[(unet.std_ == noise[0])][metric],
            marker=noise[2],
            label=fr'$\\sigma={noise[1]}$ m',
            markersize=6
        )

        if save_:
            handles.append(h_)
            labels.append(f'$\\sigma={noise[1]}$ m')

    save_ = False

for a in ax.flatten():
    a.set_xticks(x)
    a.set_xlabel('Ensemble size $n$')

ax[0].set_title('UNet')
ax[0].set_ylabel('RMSE (cm)')
ax[1].set_title('UNet')
ax[1].set_ylabel(r'$\lambda$ (km)')

fig.legend(
    handles, labels, loc='lower center', bbox_to_anchor=(0.5, -.075), ncol=4,
)
fig.savefig(
    f'{target_path}/Figure_4_ensemble_inference.{image_format}', dpi=dpi,
)
fig.tight_layout()